# GAIN - Polydispersity - DM4 (Friendly): nanoparticle size analysis from Gatan `.dm4` files

Same analysis as *GAIN - Polydispersity - DM4*, driven from a **three-step panel** instead of variables in the code.

**How to use**: run all cells (*Run > Run All Cells*); the panel opens in the last cell.
1. **Data**: choose a file or a folder. The panel reads the calibration and the acquisition mode from the metadata and tells you whether the images look usable for counting.
2. **Settings**: change the detection parameters on the left and see the result on one image on the right, updated as you go (a reduced copy of the image is used for speed). Choose which size measure the statistics use.
3. **Run**: choose what to save, check the summary of the settings and start. You get the statistics table, the pooled histogram and, on disk, overlay, histogram and CSV for every image plus a log of the parameters.

**Size measures**: equivalent diameter, Feret max / min, Martin diameter, aspect ratio (the CSV always contains all of them).

**Pipeline**: Otsu threshold (direction from bright-field / dark-field) -> optional watershed split of touching particles -> measurement -> statistics (mean, median, std, CV, PDI).

**Limits**: Otsu + watershed works best on separated, roughly uniform particles. Strongly overlapping or stacked grains are not separated reliably: always look at the overlay.

**Dependencies**: `hyperspy`, `opencv-python-headless`, `numpy`, `scipy`, `scikit-image`, `matplotlib`, `pandas`, `ipywidgets`, `ipyfilechooser` (all in the HyperSpy bundle).


In [1]:
import os
import glob
import datetime

import cv2
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
import matplotlib.pyplot as plt
import hyperspy.api as hs
import ipywidgets as W
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output

%matplotlib inline

## 1. Loading `.dm4` files and metadata

`load_dm4_image()` reads the pixel calibration (nm/px) and the acquisition info from the file and normalizes the intensities to 8-bit. Mode guess: TEM camera and bright-field STEM -> dark particles; `DF`/`ADF`/`HAADF` signals -> bright particles. `read_dm4_header()` reads only the metadata (used by the panel to describe a file quickly).


In [2]:
UNIT_TO_NM = {
    "nm": 1.0,
    "um": 1000.0,
    "µm": 1000.0,
    "pm": 0.001,
    "a": 0.1,
    "å": 0.1,
    "angstrom": 0.1,
    "m": 1e9,
}


def normalize_to_uint8(data, low_percentile=1.0, high_percentile=99.0):
    """
    Rescales an arbitrary-range array (e.g. raw CCD counts, possibly with
    negative values after dark/gain correction) to 0-255 uint8, clipping the
    extreme percentiles first so a few outlier pixels don't compress the
    contrast of the rest of the image.
    """
    lo, hi = np.percentile(data, [low_percentile, high_percentile])
    if hi <= lo:
        lo, hi = float(data.min()), float(data.max())
    clipped = np.clip(data, lo, hi)
    scaled = (clipped - lo) / (hi - lo + 1e-12) * 255.0
    return scaled.astype(np.uint8)


def find_image_tags(original_metadata):
    """Returns the ImageTags dict of the first image stored in a DigitalMicrograph file (or {})."""
    image_list = original_metadata.get("ImageList", {})
    for key in sorted(image_list):
        tags = image_list[key].get("ImageTags") if isinstance(image_list[key], dict) else None
        if isinstance(tags, dict):
            return tags
    return {}


def classify_darkfield(signal_name):
    """
    Guesses bright-field vs dark-field from the acquisition signal name.
    TEM CCD cameras (e.g. 'Orius SC200') and 'BF' STEM signals -> False
    (particles darker than background). STEM signals containing 'DF'
    ('DF', 'ADF', 'HAADF') -> True (particles brighter than background).
    """
    if not signal_name:
        return False
    return "DF" in str(signal_name).upper()


def load_dm4_image(path, signal_index=0):
    """
    Loads a Gatan DigitalMicrograph .dm4/.dm3 file with HyperSpy and returns
    (gray_uint8, nm_per_px, is_darkfield, info). Calibration and acquisition
    mode are read from the file's metadata -- no scale bar detection needed.
    If the file contains multiple signals, the one at signal_index is used
    (default: the first).
    """
    result = hs.load(path)
    signals = result if isinstance(result, list) else [result]
    if len(signals) > 1:
        print("Note: " + str(len(signals)) + " signals found in '" + path +
              "', using index " + str(signal_index) +
              " (" + str(signals[signal_index].metadata.General.title) + ")")
    s = signals[signal_index]

    if s.axes_manager.signal_dimension != 2:
        raise ValueError("'" + path + "' does not contain a 2D image (signal dimension = " +
                         str(s.axes_manager.signal_dimension) + ").")

    ax = s.axes_manager.signal_axes[0]
    units = str(ax.units).strip().replace("μ", "µ")
    factor = UNIT_TO_NM.get(units.lower())
    if factor is None:
        raise ValueError(
            "No usable spatial calibration in '" + path + "': axis units = '" + units +
            "' (diffraction pattern, or an uncalibrated image?). Cannot convert pixels to nm."
        )
    nm_per_px = float(ax.scale) * factor

    tags = find_image_tags(s.original_metadata.as_dictionary())
    microscope = tags.get("Microscope Info", {})
    databar = tags.get("DataBar", {})
    detector = databar.get("Signal Name") or databar.get("Device Name") or s.metadata.General.title
    is_darkfield = classify_darkfield(detector)

    gray = normalize_to_uint8(s.data)

    info = {
        "detector": detector,
        "illumination_mode": microscope.get("Illumination Mode"),
        "magnification": microscope.get("Formatted Indicated Mag"),
        "voltage": microscope.get("Formatted Voltage"),
        "raw_dtype": str(s.data.dtype),
        "raw_min": float(s.data.min()),
        "raw_max": float(s.data.max()),
    }
    return gray, nm_per_px, is_darkfield, info


def read_dm4_header(path, signal_index=0):
    """Reads only the metadata of a .dm4/.dm3 file (no pixel data): returns a dict with nm_per_px, size, fov_nm, detector, magnification, voltage, is_darkfield."""
    result = hs.load(path, lazy=True)
    signals = result if isinstance(result, list) else [result]
    s = signals[signal_index]
    ax = s.axes_manager.signal_axes[0]
    units = str(ax.units).strip().replace("μ", "µ")
    factor = UNIT_TO_NM.get(units.lower())
    if factor is None:
        raise ValueError("no usable spatial calibration (axis units = '" + units + "')")
    nm_per_px = float(ax.scale) * factor
    shape = s.axes_manager.signal_shape      # (width, height)
    tags = find_image_tags(s.original_metadata.as_dictionary())
    microscope = tags.get("Microscope Info", {})
    databar = tags.get("DataBar", {})
    detector = databar.get("Signal Name") or databar.get("Device Name") or s.metadata.General.title
    return {
        "nm_per_px": nm_per_px, "width": int(shape[0]), "height": int(shape[1]),
        "fov_nm": nm_per_px * min(shape), "detector": detector,
        "magnification": microscope.get("Formatted Indicated Mag"), "voltage": microscope.get("Formatted Voltage"),
        "is_darkfield": classify_darkfield(detector),
    }


def read_dm4_header(path, signal_index=0):
    """Reads only the metadata of a .dm4/.dm3 file (no pixel data): nm_per_px, size, fov_nm, detector, magnification, voltage, is_darkfield."""
    result = hs.load(path, lazy=True)
    signals = result if isinstance(result, list) else [result]
    s = signals[signal_index]
    ax = s.axes_manager.signal_axes[0]
    units = str(ax.units).strip().replace("μ", "µ")
    factor = UNIT_TO_NM.get(units.lower())
    if factor is None:
        raise ValueError("no usable spatial calibration (axis units = '" + units + "')")
    nm_per_px = float(ax.scale) * factor
    shape = s.axes_manager.signal_shape      # (width, height)
    tags = find_image_tags(s.original_metadata.as_dictionary())
    microscope = tags.get("Microscope Info", {})
    databar = tags.get("DataBar", {})
    detector = databar.get("Signal Name") or databar.get("Device Name") or s.metadata.General.title
    return {
        "nm_per_px": nm_per_px, "width": int(shape[0]), "height": int(shape[1]),
        "fov_nm": nm_per_px * min(shape), "detector": detector,
        "magnification": microscope.get("Formatted Indicated Mag"), "voltage": microscope.get("Formatted Voltage"),
        "is_darkfield": classify_darkfield(detector),
    }

## 2. Segmentation, measurement, statistics, figures

The part that does the work. `analyze_dm4_image(path, cfg)` takes a dictionary of settings (the panel builds it), converts the lengths in nm to pixels with each file's own calibration and never crashes on zero detected particles.


In [3]:
MIN_DIAM_PX_FLOOR = 6   # never accept regions smaller than this, however small the nm limit is (pure noise)

# measure key -> (name, unit)
MEASURES = {
    "eq_diam_nm": ("Equivalent diameter", "nm"),
    "feret_max_nm": ("Feret diameter, maximum", "nm"),
    "feret_min_nm": ("Feret diameter, minimum", "nm"),
    "martin_nm": ("Martin diameter (mean over directions)", "nm"),
    "aspect_ratio": ("Aspect ratio (Feret max / Feret min)", ""),
}


def measure_label(key):
    name, unit = MEASURES[key]
    return name + (" (" + unit + ")" if unit else "")


def segment_and_split_particles(gray_img, min_peak_distance=15, bright_particles=False, separate_touching=True):
    """
    Otsu segmentation. Dark particles on a bright background (bright_particles=False) or the opposite.
    With separate_touching=True touching particles are split by a watershed on the distance transform
    (min_peak_distance = minimum distance in pixels between particle centres); otherwise every connected blob is one particle.
    Returns (labels, mask).
    """
    blur = cv2.GaussianBlur(gray_img, (5, 5), 0)
    thresh_type = cv2.THRESH_BINARY if bright_particles else cv2.THRESH_BINARY_INV
    _, th = cv2.threshold(blur, 0, 255, thresh_type + cv2.THRESH_OTSU)

    kernel = np.ones((3, 3), np.uint8)
    th_clean = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=1)
    th_clean = cv2.morphologyEx(th_clean, cv2.MORPH_CLOSE, kernel, iterations=1)
    mask = th_clean > 0

    if not separate_touching:
        labels, _ = ndi.label(mask)
        return labels, mask

    dist = ndi.distance_transform_edt(mask)
    coords = peak_local_max(dist, min_distance=int(max(1, min_peak_distance)), labels=mask, exclude_border=False)
    markers_mask = np.zeros(dist.shape, dtype=bool)
    markers_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(markers_mask)
    return watershed(-dist, markers, mask=mask), mask


_DIRS = np.stack([np.cos(np.deg2rad(np.arange(0, 180, 1.0))), np.sin(np.deg2rad(np.arange(0, 180, 1.0)))], axis=1)


def feret_diameters(coords_rc):
    """(Feret max, Feret min) in pixels from the pixel coordinates (row, col) of a region: caliper widths of its convex hull."""
    pts = np.ascontiguousarray(coords_rc[:, ::-1].astype(np.float32))
    hull = cv2.convexHull(pts)[:, 0, :]
    widths = (hull @ _DIRS.T).max(axis=0) - (hull @ _DIRS.T).min(axis=0) + 1.0
    return float(widths.max()), float(widths.min())


def martin_diameter(mask, step=2.0):
    """
    Martin diameter in pixels, averaged over the directions: length of the chord that splits the area of the particle in two
    equal halves. The mask is rotated (bilinear), so the result does not depend on the pixel grid.
    """
    m = np.pad(mask.astype(np.float32), 2)
    h, w = m.shape
    n = int(np.ceil(np.hypot(h, w))) + 4
    canvas = np.zeros((n, n), np.float32)
    y0, x0 = (n - h) // 2, (n - w) // 2
    canvas[y0:y0 + h, x0:x0 + w] = m
    c = (n / 2.0, n / 2.0)
    chords = []
    for ang in np.arange(0, 180, step):
        rot = cv2.warpAffine(canvas, cv2.getRotationMatrix2D(c, float(ang), 1.0), (n, n), flags=cv2.INTER_LINEAR)
        rows = rot.sum(axis=1)
        cum = np.cumsum(rows)
        i = int(np.searchsorted(cum, cum[-1] / 2.0))
        chords.append(rows[max(i - 1, 0):i + 2].mean())
    return float(np.mean(chords))


def measure_particles(labels_ws, nm_per_px, min_diam_px=25, min_circularity=0.5, with_martin=True):
    """
    Measures every watershed region and drops those that are too small or not circular enough (noise fragments, edge artifacts).
    Returns a list of dicts with: label, area_px, circularity, solidity, eq_diam_nm, feret_max_nm, feret_min_nm, martin_nm,
    aspect_ratio, eq_diam_px, centroid (row, col), bbox, contour (x, y points, for the overlay).
    """
    particles = []
    for p in regionprops(labels_ws):
        area = p.area
        perim = p.perimeter if p.perimeter > 0 else 1
        circularity = 4 * np.pi * area / (perim ** 2)
        eq_diam_px = 2 * np.sqrt(area / np.pi)
        if eq_diam_px < min_diam_px or circularity < min_circularity:
            continue
        fmax, fmin = feret_diameters(p.coords)
        r0, c0 = p.bbox[0], p.bbox[1]
        cnts, _ = cv2.findContours(np.pad(p.image.astype(np.uint8), 1), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contour = max(cnts, key=cv2.contourArea) - 1 + np.array([[c0, r0]])
        particles.append({
            "label": p.label, "area_px": area, "circularity": circularity, "solidity": p.solidity,
            "eq_diam_px": eq_diam_px, "eq_diam_nm": eq_diam_px * nm_per_px,
            "feret_max_nm": fmax * nm_per_px, "feret_min_nm": fmin * nm_per_px,
            "martin_nm": (martin_diameter(p.image) * nm_per_px) if with_martin else float("nan"),
            "aspect_ratio": fmax / max(fmin, 1e-6),
            "centroid": p.centroid, "bbox": p.bbox, "contour": contour,
        })
    return particles


def touches_border(p, shape):
    """True if the particle's bounding box touches the image edge (the particle is cut by the frame)."""
    min_row, min_col, max_row, max_col = p["bbox"]
    return min_row == 0 or min_col == 0 or max_row >= shape[0] or max_col >= shape[1]


def polydispersity_stats(values):
    """
    Statistics of a non-empty array of sizes. PDI = (sigma/mean)^2, the DLS convention: a dimensionless number, lower = more
    monodisperse (e.g. PDI < 0.05 - 0.1 in TEM). The keys end in _nm for historical reasons; for the aspect ratio the values are unitless.
    """
    values = np.asarray(values, dtype=float)
    mean_d = values.mean()
    std_d = values.std(ddof=1) if values.size > 1 else 0.0
    return {
        "n": len(values), "mean_nm": mean_d, "median_nm": float(np.median(values)), "std_nm": std_d,
        "cv_percent": std_d / mean_d * 100, "pdi": (std_d / mean_d) ** 2,
        "min_nm": float(values.min()), "max_nm": float(values.max()),
    }


def draw_overlay(color_img, particles, border=None, oversized=None, style="circles"):
    """Kept: red (circle with the equivalent diameter, or the true contour). Cut by the border: cyan. Oversized: magenta."""
    overlay = color_img.copy()
    thick = max(2, int(round(min(overlay.shape[:2]) / 700)))

    def draw(group, color, centre_dot=False):
        for p in group:
            if style == "contours":
                cv2.drawContours(overlay, [p["contour"]], -1, color, thick)
            else:
                cy, cx = p["centroid"]
                cv2.circle(overlay, (int(round(cx)), int(round(cy))), int(round(p["eq_diam_px"] / 2)), color, thick)
            if centre_dot:
                cy, cx = p["centroid"]
                cv2.circle(overlay, (int(round(cx)), int(round(cy))), 1, (0, 255, 0), thick)

    draw(oversized or [], (255, 0, 255))
    draw(border or [], (255, 255, 0))
    draw(particles, (0, 0, 255), centre_dot=True)
    return overlay


def plot_histogram(values, stats, measure="eq_diam_nm", out_path=None, material="nanoparticles", show=True, bins=30):
    plt.figure(figsize=(7, 5))
    plt.hist(values, bins=bins, color="#4C72B0", edgecolor="white")
    unit = MEASURES[measure][1]
    plt.axvline(stats["mean_nm"], color="crimson", linestyle="--", linewidth=1.5,
                label="Mean = " + format(stats["mean_nm"], ".2f") + (" " + unit if unit else ""))
    plt.xlabel(measure_label(measure))
    plt.ylabel("Count")
    plt.title(
        "Size distribution " + material + " (N=" + str(stats["n"]) + ")\n"
        "Std dev = " + format(stats["std_nm"], ".2f") + (" " + unit if unit else "") +
        ", CV = " + format(stats["cv_percent"], ".1f") + "%, PDI = " + format(stats["pdi"], ".4f")
    )
    plt.legend()
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=150)
    if show:
        plt.show()
    else:
        plt.close()


CONTRAST_OVERRIDE = {"auto": None, "dark": False, "bright": True}


def analyze_dm4_image(path, cfg, loaded=None, with_martin=True):
    """
    Full pipeline on one .dm4 file with the settings in `cfg` (see DEFAULTS in the panel). Returns a dict: path, nm_per_px, fov_nm,
    is_darkfield, detector, info, gray, mask, particles (kept), border (cut by the image edge), oversized, stats (of the chosen
    measure, None if no particle), overlay, warnings (list of text). Never raises for zero particles.
    """
    if loaded is None:
        gray, nm_per_px, is_darkfield, info = load_dm4_image(path)
    else:
        gray, nm_per_px, is_darkfield, info = loaded   # e.g. a downscaled copy, for the live preview
    override = CONTRAST_OVERRIDE[cfg["contrast"]]
    if override is not None:
        is_darkfield = override

    min_diam_px = max(cfg["min_diam_nm"] / nm_per_px, MIN_DIAM_PX_FLOOR)
    min_peak = max(cfg["min_peak_distance_nm"] / nm_per_px, 3)
    labels_ws, mask = segment_and_split_particles(gray, min_peak_distance=min_peak, bright_particles=is_darkfield,
                                                  separate_touching=cfg["separate_touching"])
    particles = measure_particles(labels_ws, nm_per_px, min_diam_px=min_diam_px, min_circularity=cfg["min_circularity"],
                                  with_martin=with_martin)

    border = []
    if cfg["exclude_border"] and particles:
        border = [p for p in particles if touches_border(p, gray.shape)]
        particles = [p for p in particles if not touches_border(p, gray.shape)]

    oversized = []
    if cfg["max_diameter_ratio"] and particles:
        limit = cfg["max_diameter_ratio"] * np.mean([p["eq_diam_nm"] for p in particles])
        oversized = [p for p in particles if p["eq_diam_nm"] > limit]
        particles = [p for p in particles if p["eq_diam_nm"] <= limit]

    fov_nm = nm_per_px * min(gray.shape)
    warnings = []
    stats = overlay = None
    if particles:
        stats = polydispersity_stats([p[cfg["measure"]] for p in particles])
        med_diam = float(np.median([p["eq_diam_nm"] for p in particles]))
        if med_diam > fov_nm / 3:
            warnings.append("the particles fill more than a third of the field of view (median diameter " + format(med_diam, ".0f") +
                            " nm, field " + format(fov_nm, ".0f") + " nm): too much magnification for counting")
        if len(particles) < 3:
            warnings.append("only " + str(len(particles)) + " particle(s): statistics are weak")
        if border and len(border) > len(particles):
            warnings.append("more particles cut by the border (" + str(len(border)) + ") than counted (" + str(len(particles)) + ")")
    else:
        warnings.append("no particles detected: check the particle contrast (Main), the minimum diameter and the circularity (Detection)")
    if particles or border or oversized:
        overlay = draw_overlay(cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR), particles, border=border, oversized=oversized,
                               style=cfg["overlay_style"])

    return {
        "path": path, "nm_per_px": nm_per_px, "fov_nm": fov_nm, "is_darkfield": is_darkfield, "detector": info["detector"],
        "info": info, "gray": gray, "mask": mask, "particles": particles, "border": border, "oversized": oversized,
        "stats": stats, "overlay": overlay, "warnings": warnings, "min_diam_px_used": min_diam_px,
    }


def particles_table(result):
    """One row per particle kept, with every measure (used for the CSV)."""
    cols = ["eq_diam_nm", "feret_max_nm", "feret_min_nm", "martin_nm", "aspect_ratio", "circularity", "solidity", "area_px"]
    rows = []
    for i, p in enumerate(result["particles"], 1):
        row = {"particle": i}
        row.update({c: p[c] for c in cols})
        row["centroid_row"], row["centroid_col"] = p["centroid"]
        rows.append(row)
    return pd.DataFrame(rows)


def show_results(result, cfg):
    """Metadata summary, warnings and, if particles were found, overlay + histogram; otherwise the segmentation mask."""
    info = result["info"]
    print("File: " + result["path"])
    print("Acquisition: " + str(info["illumination_mode"]) + " | camera/signal: " + str(info["detector"]) +
          " | mag: " + str(info["magnification"]) + " | " + str(info["voltage"]))
    print("Mode used: " + ("dark-field (bright particles)" if result["is_darkfield"] else "bright-field (dark particles)") +
          " | calibration: " + format(result["nm_per_px"], ".4f") + " nm/px | field of view: " + format(result["fov_nm"], ".0f") + " nm")
    print("Particles kept: " + str(len(result["particles"])) + " | cut by image border: " + str(len(result["border"])) +
          " | oversized: " + str(len(result["oversized"])))
    for w in result["warnings"]:
        print("WARNING: " + w)

    if not result["particles"]:
        plt.figure(figsize=(7, 7))
        plt.imshow(result["mask"], cmap="gray")
        plt.title("Segmentation mask (no particles passed the filters)")
        plt.axis("off")
        plt.show()
        return
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(result["overlay"], cv2.COLOR_BGR2RGB))
    plt.title("Detected particles (N=" + str(result["stats"]["n"]) + ")\nred = counted, cyan = cut by border, magenta = oversized")
    plt.axis("off")
    plt.show()
    plot_histogram([p[cfg["measure"]] for p in result["particles"]], result["stats"], measure=cfg["measure"],
                   material=cfg["material"], bins=cfg["hist_bins"])


def save_results(result, outdir, cfg):
    """Saves overlay.png, histogram.png and particles.csv of one result into outdir, as selected in the Output tab."""
    if not result["particles"]:
        return []
    os.makedirs(outdir, exist_ok=True)
    saved = []
    if cfg["save_overlay"]:
        cv2.imwrite(os.path.join(outdir, "overlay.png"), result["overlay"]); saved.append("overlay.png")
    if cfg["save_histogram"]:
        plot_histogram([p[cfg["measure"]] for p in result["particles"]], result["stats"], measure=cfg["measure"],
                       out_path=os.path.join(outdir, "histogram.png"), material=cfg["material"], show=False, bins=cfg["hist_bins"])
        saved.append("histogram.png")
    if cfg["save_csv"]:
        particles_table(result).to_csv(os.path.join(outdir, "particles.csv"), index=False); saved.append("particles.csv")
    return saved


def list_dm4_files(folder):
    return sorted({p for pattern in ("*.dm4", "*.dm3") for p in glob.glob(os.path.join(folder, pattern))})


def summarize_results(results, measure):
    """One row per file: calibration, counts, statistics of the chosen measure, warnings."""
    rows = []
    for name, res in results.items():
        row = {"image": name, "magnification": res["info"]["magnification"], "nm_per_px": res["nm_per_px"],
               "n_cut_by_border": len(res["border"]), "n_oversized": len(res["oversized"])}
        if res["stats"] is not None:
            row.update(res["stats"])
        else:
            row["n"] = 0
        row["warnings"] = "; ".join(res["warnings"])
        rows.append(row)
    df = pd.DataFrame(rows).set_index("image").sort_index()
    return df.rename(columns={"mean_nm": "mean", "median_nm": "median", "std_nm": "std", "min_nm": "min", "max_nm": "max"})

## 3. The panel

`FriendlyPanel` builds the three steps. The defaults are in `DEFAULTS`.


In [4]:
DEFAULTS = {
    "mode": "folder", "file_path": "./Data/Gatan/NPs_0016.dm4", "folder_path": "./Data/Gatan", "preview_file": None,
    "contrast": "auto",
    "min_diam_nm": 7.0, "min_circularity": 0.5, "separate_touching": True, "min_peak_distance_nm": 5.0,
    "exclude_border": True, "max_diameter_ratio": 0.0,
    "measure": "eq_diam_nm", "hist_bins": 30, "material": "nanoparticles", "overlay_style": "circles",
    "outdir": "Result_Poly_dm4", "save_results": True, "save_overlay": True, "save_histogram": True, "save_csv": True,
    "write_log": True, "show_plots": True,
}
PREVIEW_PX = 1024      # the live preview works on a copy of the image reduced to about this size

CSS = """
<style>
.dm4f { font-family: var(--jp-ui-font-family, system-ui, -apple-system, 'Segoe UI', sans-serif); color: var(--jp-ui-font-color1, #222); max-width: 1020px; }
.dm4f .title { font-size: 20px; font-weight: 600; margin: 2px 0 0; }
.dm4f .sub { color: var(--jp-ui-font-color2, #666); font-size: 13px; margin: 0 0 6px; }
.dm4f .stepbar { border-bottom: 1px solid var(--jp-border-color2, #ddd); margin: 6px 0 10px; }
.dm4f .jupyter-button.stepbtn { background: transparent !important; border: none !important; border-bottom: 3px solid transparent !important; border-radius: 0 !important;
  color: var(--jp-ui-font-color2, #666) !important; font-size: 14px; box-shadow: none !important; height: 38px; padding: 0 18px; }
.dm4f .jupyter-button.stepbtn.active { border-bottom-color: var(--jp-brand-color1, #1976d2) !important; color: var(--jp-ui-font-color1, #222) !important; font-weight: 600; }
.dm4f .jupyter-button.stepbtn.done { color: var(--jp-success-color1, #2e7d32) !important; }
.dm4f .card { background: var(--jp-layout-color1, #fff); border: 1px solid var(--jp-border-color2, #e0e0e0); border-radius: 12px; padding: 12px 16px; margin: 6px 0; }
.dm4f .gt { font-size: 12px; font-weight: 600; letter-spacing: .04em; text-transform: uppercase; color: var(--jp-ui-font-color2, #666); margin: 2px 0 4px; }
.dm4f .lab { font-size: 13px; padding-top: 6px; }
.dm4f .hint { font-size: 12px; color: var(--jp-ui-font-color2, #777); }
.dm4f .tiles { display: grid; grid-template-columns: repeat(auto-fit, minmax(92px, 1fr)); gap: 8px; margin: 4px 0; }
.dm4f .tile { background: var(--jp-layout-color2, #f4f4f5); border-radius: 10px; padding: 8px 12px; }
.dm4f .tile .k { font-size: 11px; color: var(--jp-ui-font-color2, #777); }
.dm4f .seg .jupyter-button:first-child { border-radius: 8px 0 0 8px !important; }
.dm4f .seg .jupyter-button:last-child { border-radius: 0 8px 8px 0 !important; }
.dm4f .tile .v { font-size: 16px; font-weight: 600; }
.dm4f .msg { border-radius: 8px; padding: 7px 12px; font-size: 13px; margin: 4px 0; }
.dm4f .msg.ok { background: rgba(46,125,50,.12); color: var(--jp-success-color1, #2e7d32); }
.dm4f .msg.warn { background: rgba(237,108,2,.13); color: var(--jp-warn-color1, #b45f06); }
.dm4f .msg.err { background: rgba(211,47,47,.12); color: var(--jp-error-color1, #c62828); }
.dm4f .seg { display: flex !important; flex-direction: row !important; flex-wrap: wrap; width: auto !important; }
.dm4f .seg .jupyter-button { width: auto !important; min-width: 64px; padding: 0 14px !important; margin: 0 !important; border-radius: 0 !important; background: var(--jp-layout-color2, #f4f4f5) !important; color: var(--jp-ui-font-color1, #222) !important; border: 1px solid var(--jp-border-color2, #ddd) !important;
  box-shadow: none !important; height: 30px; font-size: 13px; }
.dm4f .seg .jupyter-button.mod-active { background: var(--jp-brand-color1, #1976d2) !important; color: #fff !important; border-color: var(--jp-brand-color1, #1976d2) !important; }
.dm4f .jupyter-button.primary { background: var(--jp-brand-color1, #1976d2) !important; color: #fff !important; border: none !important; border-radius: 8px !important; font-weight: 600; height: 36px; }
.dm4f .jupyter-button.ghost { background: transparent !important; border: 1px solid var(--jp-border-color1, #bbb) !important; border-radius: 8px !important; height: 36px; color: var(--jp-ui-font-color1, #222) !important; }
.dm4f table.sum { border-collapse: collapse; font-size: 13px; width: 100%; }
.dm4f table.sum td { padding: 3px 8px 3px 0; border-bottom: 1px solid var(--jp-border-color3, #eee); }
.dm4f table.sum td.k { color: var(--jp-ui-font-color2, #777); width: 46%; }
</style>
"""


def _esc(t):
    return str(t).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _tiles(pairs):
    return "<div class='tiles'>" + "".join("<div class='tile'><div class='k'>" + _esc(k) + "</div><div class='v'>" + _esc(v) + "</div></div>" for k, v in pairs) + "</div>"


class FriendlyPanel:
    STEPS = ["1  Data", "2  Settings", "3  Run"]

    def __init__(self, defaults=None):
        self.cfg = dict(DEFAULTS)
        if defaults:
            self.cfg.update(defaults)
        self.step = 0
        self.results, self.summary = {}, None
        self._files, self._header, self._cache = [], None, {}
        self._busy = False
        self._lock = False
        self._build()
        self._scan()
        self.go(0)

    # ------------------------------------------------------------------ helpers
    def _bind(self, key, w):
        w.observe(lambda ch: self._set(key, ch["new"]), "value")
        return w

    def _seg(self, key, options, width="auto"):
        w = W.ToggleButtons(options=options, value=self.cfg[key], layout=W.Layout(width=width))
        w.add_class("seg")
        return self._bind(key, w)

    def _slider(self, key, lo, hi, step, fmt):
        w = W.FloatSlider(value=self.cfg[key], min=lo, max=hi, step=step, readout_format=fmt, continuous_update=False, layout=W.Layout(width="260px"))
        return self._bind(key, w)

    def _row(self, label, widget, hint=None):
        parts = [W.HTML("<div class='lab'>" + _esc(label) + "</div>", layout=W.Layout(width="190px", flex="0 0 190px")), widget]
        row = W.HBox(parts, layout=W.Layout(align_items="flex-start", margin="1px 0"))
        if hint:
            return W.VBox([row, W.HTML("<div class='hint' style='margin:-2px 0 4px 190px'>" + _esc(hint) + "</div>")])
        return row

    # ------------------------------------------------------------------ layout
    def _build(self):
        c = self.cfg
        self.head = W.HTML(CSS + "<div class='title'>Nanoparticle size analysis</div><div class='sub'>Gatan .dm4 / .dm3 files &middot; polydispersity</div>")
        self.step_btns = [W.Button(description=t, layout=W.Layout(width="auto")) for t in self.STEPS]
        for i, b in enumerate(self.step_btns):
            b.add_class("stepbtn")
            b.on_click(lambda _b, i=i: self.go(i))
        self.stepbar = W.HBox(self.step_btns)
        self.stepbar.add_class("stepbar")

        # ---- step 1: data
        self.mode = self._seg("mode", [("Folder", "folder"), ("Single file", "file")])
        self.path = W.Text(value=c["folder_path"] if c["mode"] == "folder" else c["file_path"], continuous_update=False,
                           placeholder="Path to the data", layout=W.Layout(width="100%"))
        self.path.observe(lambda ch: self._on_path(ch["new"]), "value")
        self.b_browse = W.Button(description="Browse...", layout=W.Layout(width="110px"))
        self.b_browse.add_class("ghost")
        self.b_browse.on_click(self._toggle_browser)
        self.chooser_box = W.VBox([], layout=W.Layout(display="none"))
        self.contrast = self._seg("contrast", [("Auto", "auto"), ("Dark particles", "dark"), ("Bright particles", "bright")])
        self.preview_file = W.Dropdown(options=[], layout=W.Layout(width="320px"))
        self.preview_file.observe(lambda ch: self._on_preview_file(ch["new"]), "value")
        self.row_preview_file = self._row("Preview image", self.preview_file)
        self.info = W.HTML("")
        self.card_data = W.VBox([
            W.HTML("<div class='gt'>Data</div>"),
            self._row("Analyze", self.mode),
            W.HBox([self.path, self.b_browse], layout=W.Layout(width="100%", margin="4px 0")),
            self.chooser_box,
        ])
        self.card_data.add_class("card")
        self.card_info = W.VBox([W.HTML("<div class='gt'>What was found</div>"), self.info, self.row_preview_file, self._row("Particle contrast", self.contrast,
                                 "Auto reads it from the file: TEM camera and bright-field = dark particles; HAADF / DF = bright particles.")])
        self.card_info.add_class("card")
        self.page_data = W.VBox([self.card_data, self.card_info])

        # ---- step 2: settings + live preview
        self.w_min_diam = self._slider("min_diam_nm", 1.0, 100.0, 0.5, ".1f")
        self.w_circ = self._slider("min_circularity", 0.0, 1.0, 0.05, ".2f")
        self.w_split = self._seg("separate_touching", [("Yes", True), ("No", False)])
        self.w_peak = self._slider("min_peak_distance_nm", 1.0, 100.0, 0.5, ".1f")
        self.row_peak = self._row("Min. centre distance (nm)", self.w_peak, "About the radius of the smallest particle.")
        self.w_border = self._seg("exclude_border", [("Yes", True), ("No", False)])
        self.w_over = self._seg("max_diameter_ratio", [("Off", 0.0), ("2x mean", 2.0), ("2.5x", 2.5), ("3x", 3.0)])
        self.w_measure = W.Dropdown(options=[(measure_label(k), k) for k in MEASURES], value=c["measure"], layout=W.Layout(width="300px"))
        self._bind("measure", self.w_measure)
        self.w_style = self._seg("overlay_style", [("Circles", "circles"), ("Contours", "contours")])
        self.w_bins = W.BoundedIntText(value=c["hist_bins"], min=3, max=200, layout=W.Layout(width="80px"))
        self._bind("hist_bins", self.w_bins)

        gt = lambda t: W.HTML("<div class='gt'>" + t + "</div>")
        self.card_detect = W.VBox([
            gt("Detection"),
            self._row("Minimum diameter (nm)", self.w_min_diam, "Smaller regions are noise. In nm: independent of the magnification."),
            self._row("Minimum circularity", self.w_circ, "1 = perfect circle. Lower it for irregular or faceted particles."),
            self._row("Split touching particles", self.w_split),
            self.row_peak,
            self._row("Exclude cut particles", self.w_border, "Particles touching the image border are not counted."),
            self._row("Exclude oversized", self.w_over, "Drops merged clusters (diameter above this multiple of the mean)."),
        ])
        self.card_detect.add_class("card")
        self.card_measure = W.VBox([
            gt("Measure"),
            self._row("Size measure", self.w_measure, "Used for the statistics and the histogram. The CSV always has all the measures."),
            self._row("Histogram bins", self.w_bins),
            self._row("Overlay style", self.w_style),
        ])
        self.card_measure.add_class("card")
        self.prev_status = W.HTML("")
        self.prev_img = W.Image(value=b"", format="png", layout=W.Layout(width="100%", max_width="520px"))
        self.prev_tiles = W.HTML("")
        self.prev_msgs = W.HTML("")
        self.b_prev = W.Button(description="Refresh preview", layout=W.Layout(width="150px"))
        self.b_prev.add_class("ghost")
        self.b_prev.on_click(lambda _b: self._preview(force=True))
        self.card_prev = W.VBox([gt("Preview on one image"), self.prev_status, self.prev_img, self.prev_tiles, self.prev_msgs, self.b_prev],
                                layout=W.Layout(flex="1 1 46%", min_width="320px"))
        self.card_prev.add_class("card")
        left = W.VBox([self.card_detect, self.card_measure], layout=W.Layout(flex="1 1 52%", min_width="360px"))
        self.page_set = W.HBox([left, self.card_prev], layout=W.Layout(flex_flow="row wrap", align_items="flex-start", width="100%"))

        # ---- step 3: run
        self.w_save = self._seg("save_results", [("Yes", True), ("No", False)])
        self.w_outdir = W.Text(value=c["outdir"], continuous_update=False, layout=W.Layout(width="320px"))
        self._bind("outdir", self.w_outdir)
        cb = lambda key, text: self._bind(key, W.Checkbox(value=c[key], description=text, indent=False, layout=W.Layout(width="auto", margin="0 14px 0 0")))
        self.w_saveopts = W.HBox([cb("save_overlay", "Overlay"), cb("save_histogram", "Histogram"), cb("save_csv", "CSV tables"), cb("write_log", "Run log")],
                                 layout=W.Layout(flex_flow="row wrap"))
        self.w_show = self._seg("show_plots", [("All figures", True), ("Summary only", False)])
        self.row_outdir = self._row("Output folder", self.w_outdir, "Single file: results directly here. Folder: one subfolder per image plus summary.csv, combined histogram and run_log.txt.")
        self.row_saveopts = self._row("Save", self.w_saveopts)
        self.card_out = W.VBox([gt("Results"), self._row("Save to disk", self.w_save), self.row_outdir, self.row_saveopts, self._row("Show in notebook", self.w_show)])
        self.card_out.add_class("card")
        self.summary_html = W.HTML("")
        self.card_sum = W.VBox([gt("Summary of the settings"), self.summary_html])
        self.card_sum.add_class("card")
        self.run_msg = W.HTML("")
        self.b_run = W.Button(description="Run analysis", layout=W.Layout(width="190px"))
        self.b_run.add_class("primary")
        self.b_run.on_click(self._on_run)
        self.page_run = W.VBox([self.card_out, self.card_sum, self.run_msg, self.b_run])

        # ---- navigation
        self.b_back = W.Button(description="Back", layout=W.Layout(width="110px"))
        self.b_back.add_class("ghost")
        self.b_back.on_click(lambda _b: self.go(self.step - 1))
        self.b_next = W.Button(description="Next", layout=W.Layout(width="110px"))
        self.b_next.add_class("primary")
        self.b_next.on_click(lambda _b: self.go(self.step + 1))
        self.nav = W.HBox([self.b_back, self.b_next], layout=W.Layout(justify_content="space-between", margin="8px 0"))
        self.pages = [self.page_data, self.page_set, self.page_run]
        self.out = W.Output()
        self.root = W.VBox([self.head, self.stepbar, W.VBox(self.pages), self.nav])
        self.root.add_class("dm4f")
        self.out_box = W.VBox([self.out])
        self.out_box.add_class("dm4f")

    # ------------------------------------------------------------------ state changes
    def _on_path(self, value):
        key = "folder_path" if self.cfg["mode"] == "folder" else "file_path"
        self.cfg[key] = str(value).strip().strip('"')
        self.cfg["preview_file"] = None
        self._scan()

    def _on_preview_file(self, value):
        if value and not self._lock:
            self.cfg["preview_file"] = value
            self._scan()

    def _set(self, key, value):
        self.cfg[key] = value
        if key == "mode":
            self.path.value = self.cfg["folder_path"] if value == "folder" else self.cfg["file_path"]
            self.cfg["preview_file"] = None
            self._scan()
        if key == "contrast":
            self._scan()
        self._visibility()
        if self.step == 1 and key in ("min_diam_nm", "min_circularity", "separate_touching", "min_peak_distance_nm", "exclude_border",
                                       "max_diameter_ratio", "measure", "overlay_style", "contrast"):
            self._preview()
        if self.step == 2:
            self._render_summary()

    def _visibility(self):
        c = self.cfg
        self.row_peak.layout.display = "" if c["separate_touching"] else "none"
        self.row_outdir.layout.display = "" if c["save_results"] else "none"
        self.row_saveopts.layout.display = "" if c["save_results"] else "none"
        self.row_preview_file.layout.display = "" if (len(self._files) > 1) else "none"

    def _toggle_browser(self, _b):
        if self.chooser_box.children:
            self.chooser_box.children = []
            self.chooser_box.layout.display = "none"
            return
        folder = self.cfg["mode"] == "folder"
        cur = self.cfg["folder_path"] if folder else self.cfg["file_path"]
        start = cur if os.path.isdir(cur) else (os.path.dirname(os.path.abspath(cur)) if os.path.exists(os.path.dirname(os.path.abspath(cur))) else os.path.abspath("."))
        fc = FileChooser(os.path.abspath(start), show_only_dirs=folder, select_desc="Use this", change_desc="Change",
                         filter_pattern=None if folder else ["*.dm4", "*.dm3"], title="Pick the folder" if folder else "Pick the file")
        fc.layout.width = "100%"

        def chosen(_c):
            if fc.selected:
                self.path.value = fc.selected
                self.chooser_box.children = []
                self.chooser_box.layout.display = "none"
        fc.register_callback(chosen)
        self.chooser_box.children = [fc]
        self.chooser_box.layout.display = ""

    # ------------------------------------------------------------------ scanning the data
    def _scan(self):
        c = self.cfg
        if c["mode"] == "file":
            self._files = [c["file_path"]] if os.path.isfile(c["file_path"]) else []
        else:
            self._files = list_dm4_files(c["folder_path"]) if os.path.isdir(c["folder_path"]) else []
        self._header = None
        msgs = []
        if not self._files:
            what = c["file_path"] if c["mode"] == "file" else c["folder_path"]
            self.info.value = "<div class='msg err'>No .dm4 / .dm3 data found at: " + _esc(what) + "</div>"
            self.preview_file.options = []
            self._visibility()
            return
        names = [os.path.basename(f) for f in self._files]
        self._lock = True      # changing the options must not trigger a new scan
        self.preview_file.options = [(n, f) for n, f in zip(names, self._files)]
        if c["preview_file"] in self._files:
            self.preview_file.value = c["preview_file"]
        else:
            self.preview_file.value = self._files[0]
            c["preview_file"] = self._files[0]
        self._lock = False
        f = c["preview_file"]
        try:
            h = read_dm4_header(f)
            self._header = h
            guess = "Bright-field (dark particles)" if not h["is_darkfield"] else "Dark-field (bright particles)"
            used = {"auto": guess, "dark": "Dark particles (forced)", "bright": "Bright particles (forced)"}[c["contrast"]]
            html = _tiles([("Files", str(len(self._files))), ("Scale", format(h["nm_per_px"], ".3f") + " nm/px"),
                           ("Field of view", format(h["fov_nm"], ".0f") + " nm"), ("Magnification", str(h["magnification"])),
                           ("Camera / signal", str(h["detector"])), ("Contrast", used)])
            if h["fov_nm"] < 100:
                msgs.append(("warn", "The field of view is only " + format(h["fov_nm"], ".0f") + " nm: this looks like atomic resolution. Whole particles will not fit; it is not suited for counting."))
            elif h["nm_per_px"] > 3:
                msgs.append(("warn", "Low resolution (" + format(h["nm_per_px"], ".1f") + " nm/px): small particles will be only a few pixels wide."))
            else:
                msgs.append(("ok", "Calibration read from the metadata. The scale and the field of view are in the usual range for counting."))
            self.info.value = html + "".join("<div class='msg " + k + "'>" + _esc(t) + "</div>" for k, t in msgs)
        except Exception as exc:
            self.info.value = "<div class='msg err'>Cannot read the metadata of " + _esc(os.path.basename(f)) + ": " + _esc(exc) + "</div>"
        self._visibility()

    # ------------------------------------------------------------------ steps
    def go(self, n):
        n = max(0, min(2, n))
        if n > 0 and not self._files:
            self.step = 0
            n = 0
        self.step = n
        for i, (p, b) in enumerate(zip(self.pages, self.step_btns)):
            p.layout.display = "" if i == n else "none"
            for cls in ("active", "done"):
                b.remove_class(cls)
            if i == n:
                b.add_class("active")
            elif i < n:
                b.add_class("done")
        self.b_back.layout.visibility = "hidden" if n == 0 else "visible"
        self.b_next.layout.display = "none" if n == 2 else ""
        self._visibility()
        if n == 1:
            self._preview()
        if n == 2:
            self._render_summary()

    def _render_summary(self):
        c = self.cfg
        n = len(self._files)
        rows = [("Data", ("Folder: " if c["mode"] == "folder" else "File: ") + _esc(c["folder_path"] if c["mode"] == "folder" else c["file_path"]) + " (" + str(n) + " file" + ("s" if n != 1 else "") + ")"),
                ("Particle contrast", {"auto": "Auto (metadata)", "dark": "Dark particles", "bright": "Bright particles"}[c["contrast"]]),
                ("Minimum diameter / circularity", format(c["min_diam_nm"], "g") + " nm / " + format(c["min_circularity"], "g")),
                ("Split touching particles", "Yes, min. centre distance " + format(c["min_peak_distance_nm"], "g") + " nm" if c["separate_touching"] else "No"),
                ("Exclude cut / oversized", ("Yes" if c["exclude_border"] else "No") + " / " + ("off" if not c["max_diameter_ratio"] else format(c["max_diameter_ratio"], "g") + "x mean")),
                ("Size measure", measure_label(c["measure"])),
                ("Output", _esc(os.path.abspath(c["outdir"])) if c["save_results"] else "Nothing is saved")]
        self.summary_html.value = "<table class='sum'>" + "".join("<tr><td class='k'>" + k + "</td><td>" + v + "</td></tr>" for k, v in rows) + "</table>"
        msg = ""
        if not self._files:
            msg = "<div class='msg err'>No data selected: go back to step 1.</div>"
        elif c["save_results"] and not (c["save_overlay"] or c["save_histogram"] or c["save_csv"] or c["write_log"]):
            msg = "<div class='msg warn'>Saving is on but every output type is off: nothing will be written.</div>"
        self.run_msg.value = msg

    # ------------------------------------------------------------------ live preview
    def _load_small(self, path):
        if path not in self._cache:
            gray, nm, df, info = load_dm4_image(path)
            f = max(1, int(np.ceil(max(gray.shape) / PREVIEW_PX)))
            small = cv2.resize(gray, (gray.shape[1] // f, gray.shape[0] // f), interpolation=cv2.INTER_AREA) if f > 1 else gray
            self._cache[path] = (small, nm * f, df, info, f)
            if len(self._cache) > 4:
                self._cache.pop(next(iter(self._cache)))
        return self._cache[path]

    def _preview(self, force=False):
        if self._busy or not self._files:
            return
        path = self.cfg["preview_file"] or self._files[0]
        self._busy = True
        self.prev_status.value = "<div class='hint'>Computing...</div>"
        try:
            small, nm_small, df, info, f = self._load_small(path)
            res = analyze_dm4_image(path, self.cfg, loaded=(small, nm_small, df, info), with_martin=(self.cfg["measure"] == "martin_nm"))
            img = res["overlay"] if res["overlay"] is not None else cv2.cvtColor((res["mask"] * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
            self.prev_img.value = cv2.imencode(".png", img)[1].tobytes()
            st, m = res["stats"], self.cfg["measure"]
            unit = MEASURES[m][1]
            if st:
                tiles = [("Particles", str(st["n"])), ("Mean", format(st["mean_nm"], ".2f") + (" " + unit if unit else "")),
                         ("CV", format(st["cv_percent"], ".1f") + " %"), ("PDI", format(st["pdi"], ".3f"))]
            else:
                tiles = [("Particles", "0")]
            self.prev_tiles.value = _tiles(tiles)
            msgs = ["<div class='msg warn'>" + _esc(w) + "</div>" for w in res["warnings"]]
            note = "Preview on " + os.path.basename(path) + (" (reduced " + str(f) + "x for speed; the final run uses the full resolution)" if f > 1 else "")
            self.prev_msgs.value = "".join(msgs)
            self.prev_status.value = "<div class='hint'>" + _esc(note) + "</div>"
        except Exception as exc:
            self.prev_status.value = "<div class='msg err'>" + _esc(type(exc).__name__ + ": " + str(exc)) + "</div>"
        finally:
            self._busy = False

    # ------------------------------------------------------------------ run
    def _on_run(self, _b):
        with self.out:
            clear_output(wait=True)
            if not self._files:
                print("No data selected."); return
            self.b_run.disabled = True
            try:
                self._run_all()
            except Exception as exc:
                print("ERROR: " + type(exc).__name__ + ": " + str(exc))
            finally:
                self.b_run.disabled = False

    def _run_all(self):
        c = dict(self.cfg)
        files = self._files
        folder_mode = c["mode"] == "folder"
        save, base = c["save_results"], c["outdir"]
        log = ["GAIN Polydispersity - DM4 Friendly run log", "Date: " + datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
               "Mode: " + c["mode"] + " | files: " + str(len(files)), "", "Settings:"] + ["  " + k + " = " + str(v) for k, v in c.items()] + [""]
        bar = W.IntProgress(min=0, max=len(files), description="Files:", layout=W.Layout(width="60%"))
        display(bar)
        results = {}
        for path in files:
            name = os.path.splitext(os.path.basename(path))[0]
            try:
                res = analyze_dm4_image(path, c)
            except Exception as exc:
                print("[SKIPPED] " + name + ": " + str(exc)); log.append("[SKIPPED] " + name + ": " + str(exc)); bar.value += 1
                continue
            results[name] = res
            s = res["stats"]
            line = name + ": " + ("N=" + str(s["n"]) + ", mean=" + format(s["mean_nm"], ".2f") + ", CV=" + format(s["cv_percent"], ".1f") + "%, PDI=" + format(s["pdi"], ".4f") if s else "0 particles")
            print(line)
            log.append(line + " | cut by border: " + str(len(res["border"])) + " | oversized: " + str(len(res["oversized"])))
            for w in res["warnings"]:
                print("   WARNING: " + w); log.append("   WARNING: " + w)
            if c["show_plots"]:
                show_results(res, c)
            if save:
                save_results(res, os.path.join(base, name) if folder_mode else base, c)
            bar.value += 1
        self.results = results
        if not results:
            print("No file could be analyzed."); return
        self.summary = summarize_results(results, c["measure"])
        print("\nSummary (" + measure_label(c["measure"]) + "):")
        display(self.summary)
        if folder_mode:
            groups = [[p[c["measure"]] for p in r["particles"]] for r in results.values() if r["particles"]]
            if groups:
                pooled = np.concatenate(groups)
                pstats = polydispersity_stats(pooled)
                print("Combined distribution (all files pooled):")
                plot_histogram(pooled, pstats, measure=c["measure"], material=c["material"], bins=c["hist_bins"],
                               out_path=os.path.join(base, "combined_histogram.png") if save and c["save_histogram"] else None)
                display(pd.Series(pstats).to_frame().T)
                log.append("Pooled: " + str(pstats))
                if save and c["save_csv"]:
                    allp = pd.concat([particles_table(r).assign(image=n) for n, r in results.items() if r["particles"]], ignore_index=True)
                    os.makedirs(base, exist_ok=True)
                    allp.to_csv(os.path.join(base, "combined_particles.csv"), index=False)
            if save and c["save_csv"]:
                os.makedirs(base, exist_ok=True)
                self.summary.to_csv(os.path.join(base, "summary.csv"))
        if save and c["write_log"]:
            os.makedirs(base, exist_ok=True)
            with open(os.path.join(base, "run_log.txt"), "w", encoding="utf-8") as f:
                f.write("\n".join(log) + "\n")
        if save:
            print("\nResults saved in: " + os.path.abspath(base))

    def show(self):
        display(self.root, self.out_box)
        return self


def launch(defaults=None):
    """Opens the panel. Returns it: panel.cfg = settings, panel.results / panel.summary = last run."""
    return FriendlyPanel(defaults).show()

## 4. Launch

Run this cell to open the panel. After **Run analysis** the results stay available in Python: `panel.results` (one entry per image), `panel.summary` (table), `panel.cfg` (the settings used).


In [5]:
panel = launch()